In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mp
import ulmo
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import Polygon
import os

In [ ]:
import dataretrieval as dr
from dataretrieval import nwis


# In this study, we used Klamath River Basin as an example

In [ ]:
shp = gpd.read_file("klamath_river_basin.gpkg")

In [ ]:
shp.plot(facecolor='none')

# Step 1: Identify which HUC level 2 region that your targeted basin is located?

Maps for the HUC-2 region: https://www.usgs.gov/media/images/watershed-boundary-dataset-structure-visualization

In [ ]:
# selected huc region
HUC2code = '18'

### `ulmo.usgs.nwis.get_sites()` can be used to download the meta data for all available USGS sites within a specified HUC2 region

In [ ]:
sites = ulmo.usgs.nwis.get_sites(huc=HUC2code, service=None)

In [ ]:
type(sites), len(sites)

In [ ]:
sites['09527590']

### Next, we reorganize the site information from `dictionary` to `DataFrame`. 
#### It is especially important to extract the **latitude/longitude information** because we would need their location information to decide whether they are located in our targeted river basin.

In [ ]:
# pull out lat & lon from the nested dictionary
for k, v in sites.items():
    if 'location' in v.keys():
        v['longitude'] = v['location']['longitude']
        v['latitude'] = v['location']['latitude']
    else:
        v['longitude'] = np.NaN
        v['latitude'] = np.NaN

In [ ]:
site_info_df = pd.DataFrame(sites)
site_info_df = site_info_df.T

site_info_df['latitude'] = site_info_df['latitude'].astype(float)
site_info_df['longitude'] = site_info_df['longitude'].astype(float)
site_info_df.drop(columns=['location'], inplace=True)
site_info_df = site_info_df.dropna()

In [ ]:
site_info_df

# Step 2: Find all sites that are located in one basin

### First, we convert the flow site lat/lon to `Point` Shapefile

In [ ]:
# read in site information 
flow_site_gdf = gpd.GeoDataFrame(site_info_df, 
                                 geometry=gpd.points_from_xy(site_info_df.longitude, site_info_df.latitude))

### Second, we only select the sites that are located within our targeted shapefile

In [ ]:
# select sites within the domain
flow_site_gdf_sel = flow_site_gdf[flow_site_gdf.within(shp.geometry[0])]

# select sites with the correct site types
flow_site_gdf_sel = flow_site_gdf_sel[flow_site_gdf_sel['site_type']=='ST']

In [ ]:
flow_site_gdf_sel

# Step 3: It's time to download the data!

### Use functions from ulmo: *ulmo.usgs.nwis.get_site_data()*
* site_code: USGS site number (str)
* parameter_code: Variable code (str) ([Link to all USGS parameter code](https://help.waterdata.usgs.gov/codes-and-parameters/parameters), [Link to USGS parameters commonly used in hydrology](https://help.waterdata.usgs.gov/parameter_cd?group_cd=PHY))
    * 00060: Stream flow, cubic feet per second
    * 00010: Temperature, water, degrees Celsius
    * 00011: Temperature, water, degrees Fahrenheit
* statistic_code: Statistic code (str) ([Link to all USGS statistic code](https://help.waterdata.usgs.gov/stat_code))
    * 00001: Maximum
    * 00002: Minimum
    * 00003: Mean

In [ ]:
# download USGS data using ulmo

site_code = '11530500'
param_code = '00060'
stat_code  = '00003'
start_date = '1980-01-01'
end_date   = '2020-12-31'
site_data = ulmo.usgs.nwis.get_site_data(site_code=site_code,parameter_code=param_code, statistic_code=stat_code,
                                         service='daily', start=start_date,end=end_date,
                                         methods="all")


In [ ]:
site_data

In [ ]:
# When we download the data, we can collect following information
# at the same time.
# For example, sites with data, the first and last date with data,
# and the total number of days with data.
# We can use this type of information to do data filtering in the 
# next steps.
site_w_data_ls = []
start_date_ls  = []
end_date_ls    = []
number_of_days_w_data_ls = []

# if data directory does not exist,
# create a directory named "data"
if not os.path.isdir("data"):
    os.mkdir("data")

for site_code in flow_site_gdf_sel.index.values: #'15515500' # '15485500' #

    # download USGS data using ulmo
    param_code = '00060'
    stat_code  = '00003'
    start_date = '1980-01-01'
    end_date   = '2020-12-31'
    site_data = ulmo.usgs.nwis.get_site_data(site_code=site_code,parameter_code=param_code, service='daily',
                                             statistic_code=stat_code, start=start_date,end=end_date,
                                             methods="all")
    
    # convert data format to Pandas Dataframe
    # When we download the dataset, some sites have data while
    # other sites don't. Setting up "if-statement" can effectively
    # filter the sites without data
    
    if (len(site_data)>0) and ('00060:00003' in site_data.keys()):
        df = pd.DataFrame(site_data['00060'+':'+
                                    '00003']['values'])         # create dataframe
        df[site_code] = df['value'].astype(float)               # convert ['value'] to float
        df['date'] = pd.to_datetime(df['datetime'])             # convert ['datetime'] format to date and add as a new column
        df.set_index(['date'],inplace=True)                     # set date to be index
        df = df.drop(['datetime','value'],axis=1)               # delete column ['datetime']
        print("******* data for %s*******"%(site_code))
        print("# of days:%s, start date:%s, end date:%s"%(len(df), df.index[0], df.index[-1]))
        
        # summarize information for sites with data
        site_w_data_ls.append(site_code)
        start_date_ls.append(str(df.index[0])[0:10])
        end_date_ls.append(str(df.index[-1])[0:10])
        number_of_days_w_data_ls.append(len(df.dropna()))
        df.to_csv('data/%s.flow.csv'%(site_code))
    else:
        print('No data available for %s'%(site_code))

# Step 4: Data filteration

## First, we summarized the information for all sites with data into a pandas `dataframe`.

In [ ]:
df_site_summary = pd.DataFrame(np.transpose([start_date_ls,end_date_ls,number_of_days_w_data_ls]),
                               index = site_w_data_ls,
                               columns = ['start_date','end_date','number_of_days_w_data'])

In [ ]:
# convert the number_of_days_w_data to numbers
df_site_summary['number_of_days_w_data'] = df_site_summary['number_of_days_w_data'].astype(np.int32)

## Drainage areas (or catchment areas, confluence areas) are important considerations  

`ulmo` package does not provide ways for us to effectively extract information about it.

Therefore, we turn to a new package named `dataretrieval`. We imported the `nwis` module from `dataretrieval`. 

And we can use `nwis.get_info()` function to get the information for a selected USGS site

In [ ]:
drainage_area_list = []

for row in df_site_summary.iterrows():
    siteINFO = nwis.get_info(sites=str(row[0]))
    drainage_area_list.append(siteINFO[0].loc[0]['drain_area_va'])

In [ ]:
# Append the site information to the summarized site information table
df_site_summary['drain_area'] = drainage_area_list

In [ ]:
# filter sites without drainage information
df_site_summary = df_site_summary.dropna()

In [ ]:
# sort the summaried table by their drainage areas
df_site_summary.sort_values('drain_area',ascending=False)